In [ ]:
# 
#
# PURPOSE:
# 1. Load the scaled Healthy (H_scaled) and Test (T_scaled) data from
#    the artifact file created by Notebook 02.
# 2. Split the Healthy data into a training (H_train) and validation (H_val) set.
# 3. Train a CNN-Autoencoder *only* on H_train. This model will become
#    an "expert" at reconstructing "normal" flight data.
# 4. Determine an "anomaly threshold" by finding the 99th percentile of
#    reconstruction errors on the H_val set.
# 5. Save the trained Autoencoder and the threshold for the final notebook.

# --- Cell 1: Setup & Imports ---
import os, json, pickle, random, pathlib
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, UpSampling1D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2 # <-- Import l2
import matplotlib.pyplot as plt
import seaborn as sns

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# --- Cell 2: Constants & File Paths ---
# All I/O points to our central project directory
ARTIFACT_DIR = Path("Final_Run_Outputs")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Input file (from Notebook 02)
DATA_NPZ = ARTIFACT_DIR / "classification_pipeline_data.npz"

# Output files for *this* notebook
AE_MODEL_FILE = ARTIFACT_DIR / "anomaly_autoencoder.keras"
AE_THRESHOLD_FILE = ARTIFACT_DIR / "anomaly_threshold.json"

# Model constants
MAX_SEQ_LEN = 2048
N_CHANNELS = 23
SEED = 1337

# --- Seeding ---
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
rng = np.random.default_rng(SEED)

print(f"Artifacts I/O directory: {ARTIFACT_DIR.resolve()}")

In [ ]:
# --- Cell 3: Load Preprocessed Data ---
print(f"Loading data from {DATA_NPZ}...")
try:
    data_archive = np.load(DATA_NPZ, allow_pickle=True)
    # We only need H_scaled for this notebook
    H_scaled = data_archive['H_scaled']      # All healthy data (8736, 2048, 23)
    
    print(f"Loaded H_scaled: {H_scaled.shape}")
except Exception as e:
    print(f"Error loading {DATA_NPZ}: {e}")
    print("Please ensure '02_Classifier_and_Generation.ipynb' was run successfully.")

In [ ]:
# --- Cell 4: Split Healthy Data for Training/Validation ---
# We split our healthy data:
# - H_train: To train the autoencoder.
# - H_val: To set the anomaly threshold.
print("Splitting healthy data...")
indices = np.arange(len(H_scaled))
rng.shuffle(indices)
H_scaled = H_scaled[indices]

split_point = int(len(H_scaled) * 0.9) # 90% for training
H_train = H_scaled[:split_point]
H_val = H_scaled[split_point:]

print(f"H_train shape: {H_train.shape}")
print(f"H_val shape:   {H_val.shape}")

In [ ]:
# --- Cell 5: Define CNN Autoencoder ---
# This is the same AE architecture from your `Main_finished_VAE_GAN.ipynb`
def build_cnn_autoencoder(n_features=N_CHANNELS, max_len=MAX_SEQ_LEN, k_size=7, f1=128, f2=64):
    l2_reg = 1e-4
    encoder_input = Input(shape=(max_len, n_features))
    x = Conv1D(filters=f1, kernel_size=k_size, activation='relu', padding='same', kernel_regularizer=l2(l2_reg))(encoder_input)
    x = Dropout(0.2)(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)
    x = Conv1D(filters=f2, kernel_size=k_size, activation='relu', padding='same', kernel_regularizer=l2(l2_reg))(x)
    x = Dropout(0.2)(x)
    encoded = MaxPooling1D(pool_size=2, padding='same')(x)
    encoder = Model(encoder_input, encoded, name="CNN_Encoder")

    decoder_input = Input(shape=(max_len // 4, f2))
    x = Conv1D(filters=f2, kernel_size=k_size, activation='relu', padding='same', kernel_regularizer=l2(l2_reg))(decoder_input)
    x = Dropout(0.2)(x)
    x = UpSampling1D(size=2)(x)
    x = Conv1D(filters=f1, kernel_size=k_size, activation='relu', padding='same', kernel_regularizer=l2(l2_reg))(x)
    x = Dropout(0.2)(x)
    x = UpSampling1D(size=2)(x)
    decoded = Conv1D(filters=n_features, kernel_size=k_size, activation='linear', padding='same')(x) # Linear activation
    decoder = Model(decoder_input, decoded, name="CNN_Decoder")
    
    autoencoder = Model(encoder_input, decoder(encoder(encoder_input)), name="CNN_Autoencoder")
    autoencoder.compile(optimizer=Adam(learning_rate=1e-3), loss='mse')
    return autoencoder

autoencoder = build_cnn_autoencoder()
autoencoder.summary()

In [ ]:
# --- Cell 6: Train the Autoencoder ---
# This is a long-running cell.
# We train the AE *only* on H_train, with H_val as the validation set.
# The input (x) and target (y) are the same: H_train.
print("\n--- Starting Autoencoder Training ---")
callbacks = [
    ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.5, patience=5, verbose=1, min_lr=1e-6),
    EarlyStopping(monitor="val_loss", mode="min", patience=15, restore_best_weights=True, verbose=1)
]

history = autoencoder.fit(
    H_train, H_train,
    validation_data=(H_val, H_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# --- SAVE POINT ---
autoencoder.save(AE_MODEL_FILE)
print(f"Saved trained autoencoder to {AE_MODEL_FILE}")

In [ ]:
# --- Cell 7: Establish Anomaly Threshold (UQ) ---
print("Calculating reconstruction errors for HEALTHY validation set...")
# --- LOAD POINT ---
try:
    autoencoder = load_model(AE_MODEL_FILE)
    print("Loaded trained autoencoder model.")
except Exception as e:
    print(f"Could not load model: {e}. You may need to run Cell 6.")

# Get predictions (reconstructions) for the healthy validation set
H_val_pred = autoencoder.predict(H_val, batch_size=64)

# Calculate Mean Absolute Error (MAE) for each sample
# This error score is our measure of "normality" or "anomaly"
val_errors = np.mean(np.abs(H_val_pred - H_val), axis=(1, 2))

# Plot the distribution of "healthy" errors
plt.figure(figsize=(10, 6))
sns.histplot(val_errors, bins=50, kde=True)
plt.title('Distribution of Reconstruction Errors (Healthy Validation Set)')
plt.xlabel('Mean Absolute Error (Reconstruction Error)')
plt.ylabel('Frequency')
plt.savefig(ARTIFACT_DIR / "ae_healthy_error_distribution.png")
plt.show()

# Set the anomaly threshold at the 99th percentile
anomaly_threshold = np.quantile(val_errors, 0.99)
print(f"Anomaly Threshold (99th percentile): {anomaly_threshold:.6f}")

# --- SAVE POINT ---
# FIX: Cast to standard float for JSON
save_threshold = float(anomaly_threshold) 
with open(AE_THRESHOLD_FILE, "w") as f:
    json.dump({"anomaly_threshold_mae": save_threshold}, f, indent=2)
print(f"Saved anomaly threshold to {AE_THRESHOLD_FILE}")

print("\n--- Notebook 03 Complete. ---")
print("--- You may now proceed to Notebook 04 ---")